# Explore grabbing AESO data

I want to use this notebook to understand where and how I can grab data from the Alberta Electic Systems Operator (AESO). Right now my goal is just to learn a bit more about electricity markets in Alberta and also have some data ingestion demos that I can showcase. I'll start with finding and bringing in basic supply, demand, and price data.

Note that since this is an exploratory notebook I'm not going to bother with parameters for various paths and other scenarios that might differ between setups.

## Hourly Generation Metered Volumes

Gotta start somewhere and generation seems like a reasonable place. We'll pull in the historical values and then see if we can ingest updates via the API.
All of this will eventually get turned into jobs, but for now we're just getting it working and looking at the data.

In [0]:
volume_path = "/Volumes/classic_stable_ptbvhz_ip/alberta_energy/aeso_ingest"

url = "https://www.aeso.ca/assets/Uploads/data-requests/Hourly_Metered_Volumes_and_Pool_Price_and_AIL_2001-2009.csv"

from pathlib import Path

full_ingest_path = Path(volume_path) / "volume_pool-price_ail_historical"
full_ingest_path.mkdir(parents=True, exist_ok=True)

import requests
# Can extend this to loop over all the historical periods later

r = requests.get(url)
with open(full_ingest_path / "2001-2009.csv", "wb") as f:
  f.write(r.content)

In [0]:
df = spark.read.csv(str(full_ingest_path) + "/2001-2009.csv", header=True, inferSchema=True)
df.display()

Ok, so we have a date column in GMT and Local (presumably Mountain) time. Each row is an hour. We have a bunch of columns for generation sites (AIG1 etc.) with what I assume is their total net generation. In addition to that we have the actual pool price, Alberta Internal Load (AIL), the hour ahead pool price forecast, plus exports and imports to BC and Saskatchewan. Let's do some transformations to get this data in a format that's a bit easier to work with.

In [0]:
# Start with confirming understanding of the timezones and giving them better name
from pyspark.sql import functions as F
df = (
    df
    .withColumnRenamed("Date_Begin_GMT", "start_ts_utc")
    .withColumn("start_ts_mountain", F.from_utc_timestamp(F.col("start_ts_utc"), "America/Edmonton"))
)

In [0]:
# Quick check that I'm interpreting the timestamp and timezone conversion correctly
df.agg(
    F.count_if(F.col("Date_Begin_Local") != F.col("start_ts_mountain")).alias("mismatches"),
    F.count("*").alias("total")
).show()

Let's get the generation data into a more usable structure. I'm generally not going to be interested in the generation of a specific site. I'm going to want to aggregate across them, maybe after enriching site information with type of energy or where it is in the province (not available here, to be calculated elsewhere). I think I'm going to standardize on mountain time. Everything I'm going to want to do in this exploration will be happening in that region. I think it's generally safer to store in UTC and just present the results in mountain but I'm going to ignore that for now

In [0]:
ts_drop_cols = ["start_ts_utc", "Date_Begin_Local",]
ts_cols = ["start_ts_mountain"]
price_cols = ["ACTUAL_POOL_PRICE", "HOUR_AHEAD_POOL_PRICE_FORECAST", ]
interprov_cols = ["EXPORT_BC", "EXPORT_SK", "IMPORT_BC", "IMPORT_SK", ]
load_cols = ["ACTUAL_AIL"]
non_gen_cols = ts_drop_cols + ts_cols + price_cols + interprov_cols + load_cols
gen_cols = [c for c in df.columns if c not in non_gen_cols]
# Drop unused ts cols first so we don't have to repeat
df = df.drop(*ts_drop_cols)
gen_drop_cols = price_cols + interprov_cols + load_cols
generation_df = (
    df
    .drop(*gen_drop_cols)
    .melt(ids=ts_cols, values=gen_cols, variableColumnName="site", valueColumnName="generation")
)

generation_df.display()

In [0]:
# I don't need transformations for price or load let's just split them out
ail_df = (
    df
    .select("start_ts_mountain", "ACTUAL_AIL")
)
price_df = (
    df
    .select("start_ts_mountain", "ACTUAL_POOL_PRICE", "HOUR_AHEAD_POOL_PRICE_FORECAST")
)


In [0]:
interprov_df = (
    df
    .select("start_ts_mountain", "EXPORT_BC", "EXPORT_SK", "IMPORT_BC", "IMPORT_SK")
    .withColumn("net_export_bc", F.col("EXPORT_BC") - F.col("IMPORT_BC"))
    .withColumn("net_export_sk", F.col("EXPORT_SK") - F.col("IMPORT_SK"))
    .withColumn("net_export_agg", F.col("net_export_bc") + F.col("net_export_sk"))
)
interprov_df.display()

Ok, we've got some cleaned up data, let's see if I understand how things fit together. I believe that total generation in the province has to equal internal load plus exports. Let's see if the data supports that

In [0]:
gen_test_df = (
    generation_df
    .groupBy("start_ts_mountain")
    .agg(F.sum("generation").alias("gen_sum"))
    .join(ail_df, on="start_ts_mountain")
    .join(interprov_df.select("start_ts_mountain", "net_export_agg"), on="start_ts_mountain")
    .withColumn("net_load", F.col("ACTUAL_AIL") + F.col("net_export_agg"))
    .withColumn("net_load_diff", F.col("net_load") - F.col("gen_sum"))
)
gen_test_df.display()

So net load is exceeding generation. That shouldn't be the case, which points to a data quality issue somewhere. From asking Gemini a couple theories are that AESO isn't capturing generation from small facilities < 5MW, and/or Cogeneration facilities report gross load but only net generation. Either is plausible. We can do some further analysis later if I bring in more data. For now I'm going to leave this notebook alone and make a job to bring in the historical data. Then I'll come back for an exploratory notebook to grab the live data and merge it in.